# Employee Retention Dashboard Workflow — DaimonUpDown Workforce Project

## Purpose

This notebook documents the complete workflow used to prepare, validate, troubleshoot, and develop the Employee Retention Dashboard.

The workflow uses **MariaDB/SQL, the terminal, CSV data, Tableau Public, and Git/GitHub**.

The Employee Retention Dashboard uses **employee-level employment and offboarding data**. This is intentionally separate from the Workforce Dashboard, which uses assignment-level data.


## 1. Employee Retention Dashboard Process

The overall workflow was:

**MariaDB → Verify source tables → Inspect schemas → Build employee-level SQL dataset → Export TSV → Convert TSV to CSV → Verify CSV → Tableau Public → Troubleshoot → Git/GitHub → Publish**

The primary data preparation was performed with **MariaDB/SQL and terminal commands**.

Python was used only for the reusable TSV-to-CSV conversion utility. It was not used as the primary data-cleaning or analytical tool.

## 2. Employee-Level vs Assignment-Level Data

The Workforce Dashboard uses assignment-level data, where one row represents an assignment.

The Employee Retention Dashboard uses employee-level employment and offboarding information.

This distinction is important because assignment counts cannot automatically be interpreted as employee turnover or retention.

## 3. Verify MariaDB Is Running

MariaDB was verified from the terminal before querying the retention-related tables.

In [ ]:
sudo systemctl status mariadb

The MariaDB server was successfully started and reached the `ready for connections` state during the project.

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SHOW TABLES;
"

## 4. Verify Retention Source Table Counts

The three primary tables used for the Employee Retention Dashboard were checked directly in MariaDB.

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT COUNT(*) AS employees FROM employees;
SELECT COUNT(*) AS employment_history FROM employment_history;
SELECT COUNT(*) AS offboarding FROM offboarding;
"

### Verified record counts

- Employees: **1,600**
- Employment history: **1,600**
- Offboarding: **320**

These counts were verified directly against MariaDB.

## 5. Inspect the Retention Table Schemas

The employment history and offboarding schemas were inspected before constructing the dashboard dataset.

In [ ]:
sudo mariadb -e "
USE daimonupdown;
DESCRIBE employment_history;
DESCRIBE offboarding;
"

Important employment-history fields included:

- `employee_id`
- `employment_start_date`
- `employment_end_date`
- `employment_status`
- `employment_type`
- `position`
- `career_level`
- `start_reason`
- `end_reason`
- `assignment_status`

Important offboarding fields included:

- `offboarding_date`
- `departure_type`
- `leaving_reason`
- `exit_satisfaction`
- `would_recommend`
- `would_return`
- `rehire_eligible`
- `comments`

## 6. Validate Employee Relationships

Before creating the retention dataset, employment-history records were checked against the employee table.

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT COUNT(*) AS orphaned_employment_history
FROM employment_history eh
LEFT JOIN employees e
    ON eh.employee_id = e.employee_id
WHERE e.employee_id IS NULL;
"

The relationship check confirmed that employment-history records could be associated with employee records.

## 7. Create the Employee Retention Dataset

Employees were joined to employment history using `employee_id`.

Offboarding was then joined using `employment_history_id`.

A **LEFT JOIN** was intentionally used so employees without an offboarding record remain in the dataset. This is important because active employees may not have an offboarding record.

In [ ]:
sudo mariadb --batch -e "
USE daimonupdown;
SELECT
    e.employee_id,
    e.first_name,
    e.last_name,
    eh.employment_start_date,
    eh.employment_end_date,
    eh.employment_status,
    eh.employment_type,
    eh.position,
    eh.career_level,
    eh.start_reason,
    eh.end_reason,
    o.offboarding_date,
    o.departure_type,
    o.leaving_reason,
    o.exit_satisfaction,
    o.would_recommend,
    o.would_return,
    o.rehire_eligible
FROM employees e
LEFT JOIN employment_history eh
    ON e.employee_id = eh.employee_id
LEFT JOIN offboarding o
    ON eh.employment_history_id = o.employment_history_id;
" > dashboard/data/employee_retention.tsv

The SQL export creates:

`dashboard/data/employee_retention.tsv`

The TSV is an intermediate terminal export. It is later converted to CSV for Tableau.

## 8. Validate the Export Before Conversion

The exported TSV was checked from the terminal.

In [ ]:
ls -lh dashboard/data/employee_retention.tsv
head -n 2 dashboard/data/employee_retention.tsv

The export should contain column headers because the MariaDB `--batch` export was used without `--skip-column-names`.

## 9. Convert TSV to CSV

The reusable project utility `python/tsv_to_csv.py` was used to convert the MariaDB TSV export into a Tableau-ready CSV.

In [ ]:
python python/tsv_to_csv.py dashboard/data/employee_retention.tsv

The successful conversion created:

`dashboard/data/employee_retention.csv`

## 10. TSV-to-CSV Troubleshooting

The first attempt to run the conversion failed with:

`TypeError: "delimiter" must be a unicode character, not a string of length 2`

The problem was in `python/tsv_to_csv.py`.

The CSV reader had been given the two-character string `"\\t"` rather than the actual tab character `"\t"`.

The delimiter was corrected to:

```python
delimiter="\t"
```

After the correction, the conversion completed successfully.

## 11. Verify the Final CSV

The generated CSV was checked from the terminal before being used in Tableau.

In [ ]:
ls -lh dashboard/data/employee_retention.*

head -n 2 dashboard/data/employee_retention.csv

The first verified CSV row contained an `Active` employee with no offboarding record.

This confirmed that the `LEFT JOIN` preserved employees who had not been offboarded.

## 12. Employee Retention Analysis Questions

The dashboard was designed around employee-level questions:

1. How many employees are currently active?
2. How many employees have been offboarded?
3. What is the employee retention/turnover picture?
4. How often do employees leave over time?
5. What are the main reasons employees leave?
6. How does retention or turnover vary by position or other available employee attributes?

Important: a percentage calculated only from the current total employee and offboarding counts should not automatically be called an annual retention rate. A defined time period is required for an annual retention or turnover calculation.

## 13. Tableau Worksheet: Active Employees

### Purpose

Show the number of employees currently represented as active in the employment data.

### Tableau approach

Use `Employee Id` as a distinct employee count and filter the employment status to the active population.

## 14. Tableau Worksheet: Offboarded Employees

### Purpose

Show the number of employees represented in the offboarding data.

The offboarding table contains **320 records**.

## 15. Tableau Worksheet: Employee Departures Over Time

### Purpose

Show when employee departures occurred across the available reporting period.

The analysis should use `offboarding_date` for departure timing rather than assignment dates because the dashboard is measuring employee departures.

## 16. Tableau Worksheet: Leaving Reasons

### Purpose

Show the primary reasons employees left the organization.

Use `leaving_reason` from the offboarding data and count the relevant employee/offboarding records.

## 17. Tableau Worksheet: Exit Satisfaction

### Purpose

Analyze exit satisfaction among employees with available offboarding survey information.

The relevant field is `exit_satisfaction`.

NULL values should remain missing rather than being treated as zero or as a negative response.

## 18. Tableau Worksheet: Would Recommend / Would Return

The offboarding data also contains:

- `would_recommend`
- `would_return`
- `rehire_eligible`

These fields can be used to describe employee exit sentiment and rehire eligibility.

## 19. Tableau Troubleshooting

### Employee-level vs assignment-level data

The Workforce Dashboard and Employee Retention Dashboard use different grains. Assignment records must not be substituted for employee records when calculating retention or turnover.

### LEFT JOIN behavior

The retention dataset uses LEFT JOINs so active employees without offboarding records remain available to Tableau.

### Missing offboarding values

NULL offboarding fields for active employees are expected. They should not automatically be treated as errors.

### Annual retention vs overall proportion

An overall proportion of offboarded employees is not automatically an annual turnover rate. The reporting period must be explicitly defined before calculating a time-based rate.

### TSV conversion error

The TSV-to-CSV utility initially failed because the CSV reader received `"\\t"` instead of the actual tab character `"\t"`. Correcting the delimiter fixed the conversion.

## 20. Data Validation

The final Employee Retention dataset was validated at the employee level.

Verified results:

- Total employees: **1,600**
- Unique employee IDs: **1,600**
- Duplicate employee IDs: **0**
- Active employees: **1,280**
- Inactive employees: **320**
- Employees with offboarding dates: **320**
- Active employees with an offboarding date: **0**
- TSV rows: **1,600**
- Final CSV rows: **1,600**
- Successful TSV-to-CSV conversion

The employee-level grain check confirmed one row per employee in the final CSV.

The relationship checks confirmed that inactive employees were associated with offboarding records, while active employees had no offboarding date.

NULL offboarding and exit-survey values were preserved rather than interpreted as negative responses or zero values.

The resulting dataset was used as the Tableau source for the Employee Retention Dashboard.

## 21. Git and GitHub Workflow

The notebook and dashboard data preparation files are maintained in the project Git repository.

The normal workflow is:

1. Make or update project files.
2. Check `git status`.
3. Add the intended files.
4. Commit with a descriptive message.
5. Push to `origin/main`.
6. Confirm the branch is synchronized and the working tree is clean.

In [ ]:
git status
git add notebooks/04_employee_retention_dashboard.ipynb
git commit -m "Document employee retention dashboard workflow"
git push
git status

## 22. Final Employee Retention Dashboard Scope

The Employee Retention Dashboard tells this story:

**employee population → active employees → departures → leaving reasons → exit experience → retention/turnover context.**

The dashboard is based on employee-level employment and offboarding information.

## 23. Final Status

The employee retention dataset was successfully created from MariaDB, exported to TSV, converted to CSV, and verified from the terminal.

The final Tableau-ready file is:

`dashboard/data/employee_retention.csv`

The Employee Retention Dashboard can be developed and republished in Tableau Public from this verified dataset.